# MediFlow Hair — EfficientNet-B1 미세조정 5 Epoch 연장

B1의 2단계 Validation Accuracy 최고값이 마지막 10번째 Epoch에서 나왔고 Validation Loss도 끝까지 감소했습니다. 과적합 징후 없이 학습이 종료되었으므로 저장된 2단계 최고 모델에서 5 Epoch만 이어서 학습합니다.

- 시작 모델: B1, 256×256, Label Smoothing 0.05의 2단계 최고 모델
- 고정 조건: 모델 가중치, 데이터, 분할, 클래스 순서, Batch 32, Seed 42, 증강, 클래스 가중치, 학습률 1e-5, 마지막 30계층, Batch Normalization 고정, 저장된 Adam 상태
- 변경 조건: 2단계 학습 길이 10 → 최대 15 Epoch
- 선택 기준: 기존 B1과 연장 B1의 Validation Accuracy
- Test: 연장 여부를 Validation으로 결정한 뒤 선택 모델을 한 번 평가

기존 모델과 결과는 수정하거나 덮어쓰지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install seaborn scikit-learn

import hashlib
import json
import platform
import random
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('GPU가 없습니다. Colab 런타임 유형을 GPU로 변경하세요.')


## 1. 경로와 연장 설정

Drive에서 파일을 옮긴 경우에만 경로를 수정하세요.


In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
DATA_ZIP_PATH = MY_DRIVE / 'mediflow_datasets' / 'hair_clean_v1_20260907_053616.zip'
EXPECTED_DATA_SHA256 = '2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156'
SOURCE_RESULT_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / 'clean_256_efficientnetb1_ls005_20260907_111711'
SOURCE_MODEL_PATH = SOURCE_RESULT_ROOT / 'stage2_finetune_best.keras'
SOURCE_CONFIG_PATH = SOURCE_RESULT_ROOT / 'training_config.json'
SOURCE_STAGE2_HISTORY_PATH = SOURCE_RESULT_ROOT / 'stage2_history.json'
CLASS_NAMES = ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다']
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
SEED = 42
ORIGINAL_STAGE2_EPOCHS = 10
EXTRA_EPOCHS = 5
TOTAL_STAGE2_EPOCHS = ORIGINAL_STAGE2_EPOCHS + EXTRA_EPOCHS
EXPECTED_LEARNING_RATE = 1e-5
REFERENCE_B1_VAL_ACCURACY = 0.7763578295707703
REFERENCE_B1_TEST_ACCURACY = 0.7787539936102237
REFERENCE_B1_MACRO_F1 = 0.778812327205678
REFERENCE_B0_VAL_ACCURACY = 0.7739616632461548

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOCAL_ROOT = Path('/content') / f'hair_b1_extend5_{RUN_ID}'
LOCAL_ZIP_PATH = LOCAL_ROOT / DATA_ZIP_PATH.name
EXTRACT_ROOT = LOCAL_ROOT / 'dataset'
RESULT_ROOT = LOCAL_ROOT / 'results'
DRIVE_RESULT_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / f'clean_256_efficientnetb1_ls005_extended5_{RUN_ID}'
LOCAL_ROOT.mkdir(parents=True, exist_ok=False)
EXTRACT_ROOT.mkdir()
RESULT_ROOT.mkdir()
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('결과:', DRIVE_RESULT_ROOT)


## 2. 데이터와 시작 모델 검증


In [ ]:
def copy_with_sha256(source, destination):
    digest = hashlib.sha256()
    with source.open('rb') as src, destination.open('wb') as dst:
        while True:
            chunk = src.read(8 * 1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
            dst.write(chunk)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination_resolved / member.filename).resolve()
            if destination_resolved not in target.parents and target != destination_resolved:
                raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
        archive.extractall(destination_resolved)

for required in (DATA_ZIP_PATH, SOURCE_MODEL_PATH, SOURCE_CONFIG_PATH, SOURCE_STAGE2_HISTORY_PATH):
    if not required.is_file():
        raise FileNotFoundError(f'필수 파일이 없습니다: {required}')
data_sha256 = copy_with_sha256(DATA_ZIP_PATH, LOCAL_ZIP_PATH)
if data_sha256 != EXPECTED_DATA_SHA256:
    raise ValueError(f'데이터 ZIP 식별값이 다릅니다: {data_sha256}')
safe_extract(LOCAL_ZIP_PATH, EXTRACT_ROOT)
source_config = json.loads(SOURCE_CONFIG_PATH.read_text(encoding='utf-8'))
source_stage2_history = json.loads(SOURCE_STAGE2_HISTORY_PATH.read_text(encoding='utf-8'))
if source_config.get('class_names') != CLASS_NAMES or source_config.get('image_size') != list(IMAGE_SIZE):
    raise ValueError('시작 모델의 클래스 순서 또는 입력 크기가 다릅니다.')
if source_config.get('data_zip_sha256') != EXPECTED_DATA_SHA256:
    raise ValueError('시작 모델의 학습 데이터가 다릅니다.')
if source_config.get('selected_val_accuracy') != REFERENCE_B1_VAL_ACCURACY:
    raise ValueError('시작 모델의 Validation 기록이 다릅니다.')
if len(source_stage2_history['val_accuracy']) != ORIGINAL_STAGE2_EPOCHS:
    raise ValueError('기존 2단계 Epoch 기록이 다릅니다.')
print('데이터와 시작 모델 기록 검증 완료')


## 3. 동일 데이터셋 구성

모델 내부에 Rescaling(1/255)이 있으므로 외부 정규화는 적용하지 않습니다.


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
def find_dataset_root():
    matches = [path for path in EXTRACT_ROOT.rglob('augmented') if path.is_dir() and all((path / split).is_dir() for split in ('train', 'val', 'test'))]
    if len(matches) != 1:
        raise ValueError(f'augmented 폴더를 하나로 확정할 수 없습니다: {matches}')
    return matches[0]
def image_files(folder):
    return sorted(path for path in folder.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
DATA_ROOT = find_dataset_root()
counts = {split: {name: len(image_files(DATA_ROOT / split / name)) for name in CLASS_NAMES} for split in ('train', 'val', 'test')}
if any(value == 0 for split in counts.values() for value in split.values()):
    raise ValueError(f'이미지가 없는 클래스가 있습니다: {counts}')
def make_dataset(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(DATA_ROOT / split, labels='inferred', label_mode='categorical', class_names=CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED if shuffle else None).prefetch(tf.data.AUTOTUNE)
train_ds = make_dataset('train', True)
val_ds = make_dataset('val', False)
test_ds = make_dataset('test', False)
train_labels = np.concatenate([np.full(counts['train'][name], index, dtype=np.int64) for index, name in enumerate(CLASS_NAMES)])
weights = compute_class_weight(class_weight='balanced', classes=np.arange(len(CLASS_NAMES)), y=train_labels)
CLASS_WEIGHT = {index: float(weight) for index, weight in enumerate(weights)}
display(pd.DataFrame(counts).rename_axis('class'))


## 4. 저장된 Adam 상태에서 5 Epoch 연장

compile=False로 다시 설정하지 않고 저장된 optimizer 상태를 복원합니다.


In [ ]:
model = keras.models.load_model(SOURCE_MODEL_PATH, compile=True)
if model.optimizer is None:
    raise ValueError('저장된 optimizer 상태를 복원하지 못했습니다.')
learning_rate = float(tf.keras.backend.get_value(model.optimizer.learning_rate))
if not np.isclose(learning_rate, EXPECTED_LEARNING_RATE):
    raise ValueError(f'학습률이 다릅니다: {learning_rate}')
if not np.isclose(float(model.loss.label_smoothing), 0.05):
    raise ValueError(f'Label Smoothing이 다릅니다: {model.loss.label_smoothing}')
backbones = [layer for layer in model.layers if isinstance(layer, keras.Model) and 'efficientnet' in layer.name.lower()]
if len(backbones) != 1:
    raise ValueError(f'Backbone을 하나로 찾지 못했습니다: {[x.name for x in backbones]}')
backbone = backbones[0]
if any(layer.trainable for layer in backbone.layers if isinstance(layer, keras.layers.BatchNormalization)):
    raise ValueError('학습 가능한 Batch Normalization 계층이 있습니다.')
if not any(layer.trainable for layer in backbone.layers):
    raise ValueError('학습 가능한 Backbone 계층이 없습니다.')

EXTENSION_MODEL_PATH = RESULT_ROOT / 'stage2_extended_best.keras'
extension_history = model.fit(
    train_ds, validation_data=val_ds,
    initial_epoch=ORIGINAL_STAGE2_EPOCHS, epochs=TOTAL_STAGE2_EPOCHS,
    class_weight=CLASS_WEIGHT,
    callbacks=[
        keras.callbacks.ModelCheckpoint(str(EXTENSION_MODEL_PATH), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
        keras.callbacks.CSVLogger(str(RESULT_ROOT / 'extension_training_log.csv')),
    ],
    verbose=1,
)
extension_best_offset = int(np.argmax(extension_history.history['val_accuracy']))
extension_best_epoch = ORIGINAL_STAGE2_EPOCHS + extension_best_offset + 1
extension_best_val = float(extension_history.history['val_accuracy'][extension_best_offset])
print('연장 구간 Best Epoch:', extension_best_epoch)
print('연장 구간 Best Validation Accuracy:', extension_best_val)


## 5. Validation으로 연장 모델 선택 후 Test 평가


In [ ]:
if extension_best_val > REFERENCE_B1_VAL_ACCURACY:
    selected_training = 'b1_stage2_extended'
    selected_source_path = EXTENSION_MODEL_PATH
    selected_val_accuracy = extension_best_val
else:
    selected_training = 'b1_stage2_original_10_epochs'
    selected_source_path = SOURCE_MODEL_PATH
    selected_val_accuracy = REFERENCE_B1_VAL_ACCURACY
FINAL_MODEL_PATH = RESULT_ROOT / 'best_model.keras'
shutil.copy2(selected_source_path, FINAL_MODEL_PATH)
selected_model = keras.models.load_model(FINAL_MODEL_PATH, compile=False)
y_true, y_pred = [], []
for images, one_hot_labels in test_ds:
    probabilities = selected_model.predict(images, verbose=0)
    y_true.extend(np.argmax(one_hot_labels.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(probabilities, axis=1).tolist())
y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
test_accuracy = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average='macro'))
report = classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES, output_dict=True, zero_division=0)
print('Validation 기준 선택:', selected_training)
print('선택 Validation Accuracy:', selected_val_accuracy)
print('Test Accuracy:', test_accuracy)
print('Macro F1:', macro_f1)
print(classification_report(y_true, y_pred, labels=list(range(len(CLASS_NAMES))), target_names=CLASS_NAMES, digits=6, zero_division=0))


## 6. 그래프와 결과 저장


In [ ]:
extension_json = {key: [float(value) for value in values] for key, values in extension_history.history.items()}
combined_val_accuracy = source_stage2_history['val_accuracy'] + extension_json['val_accuracy']
combined_val_loss = source_stage2_history['val_loss'] + extension_json['val_loss']
epochs = np.arange(1, TOTAL_STAGE2_EPOCHS + 1)
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, combined_val_accuracy, marker='o')
axes[0].axvline(10.5, color='gray', linestyle='--')
axes[0].set(title='B1 Stage 2 Validation Accuracy', xlabel='Stage 2 Epoch', ylabel='Accuracy')
axes[0].grid(True)
axes[1].plot(epochs, combined_val_loss, marker='o')
axes[1].axvline(10.5, color='gray', linestyle='--')
axes[1].set(title='B1 Stage 2 Validation Loss', xlabel='Stage 2 Epoch', ylabel='Loss')
axes[1].grid(True)
plt.tight_layout()
plt.savefig(RESULT_ROOT / 'b1_extension_curves.png', dpi=200, bbox_inches='tight')
plt.show()
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(9, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Selected B1 Confusion Matrix')
plt.tight_layout(); plt.savefig(RESULT_ROOT / 'confusion_matrix.png', dpi=200, bbox_inches='tight'); plt.show()

config = {
    'experiment': 'hair_clean_256_efficientnetb1_extend_stage2_by_5',
    'reason': '기존 B1 최고 Validation이 마지막 Epoch이며 Validation Loss가 계속 감소함',
    'single_changed_variable': {'stage2_max_epochs': {'from': 10, 'to': 15}},
    'optimizer_state_restored': True,
    'data_zip_sha256': data_sha256, 'class_names': CLASS_NAMES, 'image_size': list(IMAGE_SIZE),
    'batch_size': BATCH_SIZE, 'seed': SEED, 'class_counts': counts, 'class_weight': CLASS_WEIGHT,
    'backbone': 'EfficientNet-B1', 'loss': {'name': 'categorical_crossentropy', 'label_smoothing': 0.05},
    'learning_rate': learning_rate, 'batch_normalization_frozen': True,
    'reference_b1': {'validation_accuracy': REFERENCE_B1_VAL_ACCURACY, 'test_accuracy': REFERENCE_B1_TEST_ACCURACY, 'macro_f1': REFERENCE_B1_MACRO_F1},
    'extension_best_epoch': extension_best_epoch, 'extension_best_val_accuracy': extension_best_val,
    'selected_training': selected_training, 'selected_val_accuracy': selected_val_accuracy,
    'test_accuracy': test_accuracy, 'macro_f1': macro_f1,
    'selection_metric': 'val_accuracy',
    'test_policy': '연장 여부를 Validation으로 결정한 뒤 선택 모델을 평가',
    'limitations': ['실제 USB 현미경 환자 데이터 없음', '원천 JSON이 없어 다중 라벨 여부 확인 불가', '사람·촬영 세션 단위 누수 확인 불가'],
    'environment': {'python': platform.python_version(), 'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__},
}
(RESULT_ROOT / 'training_config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
(RESULT_ROOT / 'extension_history.json').write_text(json.dumps(extension_json, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame(report).transpose().to_csv(RESULT_ROOT / 'classification_report.csv', encoding='utf-8-sig')
pd.DataFrame([
    {'candidate': 'B0 LS0.05', 'validation_accuracy': REFERENCE_B0_VAL_ACCURACY},
    {'candidate': 'B1 original 10', 'validation_accuracy': REFERENCE_B1_VAL_ACCURACY},
    {'candidate': 'B1 extended', 'validation_accuracy': extension_best_val},
]).to_csv(RESULT_ROOT / 'extension_comparison.csv', index=False, encoding='utf-8-sig')
if DRIVE_RESULT_ROOT.exists():
    raise FileExistsError(f'결과 폴더가 이미 있습니다: {DRIVE_RESULT_ROOT}')
DRIVE_RESULT_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RESULT_ROOT, DRIVE_RESULT_ROOT)
print('Drive 저장 완료:', DRIVE_RESULT_ROOT)
for path in sorted(DRIVE_RESULT_ROOT.iterdir()): print(' -', path.name)


## 완료 후

Drive 결과 폴더를 ZIP으로 내려받아 보내주세요. 이 결과로 공개 데이터 후보를 확정하고 패키징합니다.
